# SegCraft quick demo (YouTube -> overlay video)

Short and practical: grab a video, choose the runtime/device, and write an annotated semantic segmentation video.

## 1) Download a short YouTube clip
Use this only if you have rights to download/process the video.

In [ ]:
from segcraft.video import download_youtube

url = "https://www.youtube.com/watch?v=BHYOo3JCuvk"
video_path = download_youtube(url, "data/demo/video.mp4")
video_path

## 2) Run SegCraft on the video
The command writes original, overlay, and side-by-side comparison videos. `runtime.device: auto` uses GPU when the installed PyTorch build can see CUDA, then falls back to CPU when it cannot.

In [ ]:
from pathlib import Path
from segcraft import predict

Path("configs/local.yaml").write_text(
    "data:\n"
    "  image_size: [360, 640]\n"
    "predict:\n"
    "  input_path: data/demo/video.mp4\n"
    "  output_path: outputs/demo_predictions\n"
    "  annotate: true\n"
    "  save_video: true\n"
    "  video_max_seconds: 30\n"
    "  video_frame_stride: 2\n"
    "  preserve_audio: true\n"
    "  display:\n"
    "    show_panel: true\n"
    "    show_floating_labels: false\n"
    "    show_confidence: true\n"
    "    show_percentages: true\n"
    "    label_move_threshold: 96\n"
    "    label_smoothing: 0.85\n"
    "runtime:\n"
    "  device: auto\n"
    "  output_dir: outputs/demo\n",
    encoding="utf-8",
)

summary = predict(
    "configs/base.yaml",
    preset_path="configs/presets/cityscapes_video.yaml",
    local_path="configs/local.yaml",
)
summary["predict"]

`outputs/demo_predictions/original.mp4` is the source clip used for prediction. `overlay.mp4` is the processed semantic segmentation video. `comparison.mp4` shows both side by side. Run metadata is in `summary.json`.

## Useful knobs

- `runtime.device`: `auto`, `cuda`, or `cpu`. Keep `auto` for a friendly default.
- `predict.video_max_seconds`: limit long videos while testing.
- `predict.video_frame_stride`: `1` means every frame; larger values are faster and lower the output FPS to preserve playback speed.
- `predict.display.show_floating_labels`: turn region labels on only when you want them. The summary panel stays available either way.
- `predict.display.label_move_threshold` and `label_smoothing`: reduce label jitter when floating labels are enabled.
- `task.class_names`: the model label list. Keep it aligned with the checkpoint.